# 00 - Colab Setup And Research Configuration

Run this first. It installs dependencies, mounts Google Drive, reads your Colab secrets, logs in to Hugging Face and W&B, and writes a shared run configuration for the later notebooks.

Required Colab secrets:

- `HF_WRITE_ACCESS`
- `WANDB_KEY`

Notebook `00` itself can run on CPU. Before the main generation run in notebook `02`, start a fresh A100 runtime and rerun this setup so the pinned model dependencies are installed before Transformers is imported.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import sys, subprocess

requirements = REPO_ROOT / "requirements-colab.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
print("Installed:", requirements)

In [ ]:
from huggingface_hub import HfApi
from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, utc_timestamp

HF_TOKEN = login_huggingface("HF_WRITE_ACCESS")
api = HfApi(token=HF_TOKEN)
HF_OWNER = api.whoami()["name"]

RUN_ID = utc_timestamp()
PROJECT_NAME = "ocn_empty_negations"
HF_DATASET_PREFIX = "ocn-empty-negations"

paths = make_colab_paths(PROJECT_NAME)
run = login_wandb(
    project="ocn-empty-negations",
    name=f"setup-{RUN_ID}",
    config={"hf_owner": HF_OWNER, "run_id": RUN_ID},
)

CONFIG = {
    "run_id": RUN_ID,
    "hf_owner": HF_OWNER,
    "hf_private": False,
    "hf_prompt_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-prompts",
    "hf_generation_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-generations",
    "hf_detection_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-detection",
    "hf_main_generation_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-generations-main-gemma4-qwen35",
    "hf_main_detection_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-detection-main-gemma4-qwen35",
    "hf_main_reward_pairs_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-pairs-main-gemma4-qwen35",
    "hf_main_reward_scores_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-scores-main-gemma4-qwen35",
    "hf_reward_pairs_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-pairs",
    "hf_reward_scores_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-scores",
    "drive_project_root": str(paths.project_root),
    "drive_data_root": str(paths.data_root),
    "drive_figure_root": str(paths.figure_root),
    "default_prompt_limit": 96,
    "default_seeds": [1, 2],
}

config_path = paths.project_root / "ocn_colab_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

run.finish()